# Глава 1: Базовая структура приглашения

- [Урок](#lesson)
- [Упражнения](#exercises)
- [Пример игровой площадки](#example-playground)

## Настройка

Запустите следующую ячейку настройки, чтобы загрузить ключ API и установить вспомогательную функцию get_completion.

In [ ]:
%pip install anthropic --quiet

# Import the hints module from the utils package
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Import python's built-in regular expression library
import re
from anthropic import AnthropicBedrock

%store -r MODEL_NAME
%store -r AWS_REGION

client = AnthropicBedrock(aws_region=AWS_REGION)

def get_completion(prompt, system=''):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": prompt}
        ],
        system=system
    )
    return message.content[0].text

---

## Урок

Anthropic предлагает два API: устаревший [API дополнения текста] (https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-anthropic-claude-text-completion.html) и текущий [API сообщений] (https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-anthropic-claude-messages.html). В этом уроке мы будем использовать исключительно API сообщений.

Как минимум, для вызова Claude с использованием API сообщений требуются следующие параметры:
- `model`: [название модели API] (https://docs.aws.amazon.com/bedrock/latest/userguide/model-ids.html#model-ids-arns) модели, которую вы собираетесь вызвать.

- `max_tokens`: максимальное количество токенов, которые можно сгенерировать перед остановкой. Обратите внимание, что Клод может остановиться, не достигнув этого максимума. Этот параметр указывает только абсолютное максимальное количество генерируемых токенов. Более того, это *жесткая* остановка, то есть она может привести к тому, что Клод перестанет генерировать полуслово или полупредложение.

- `messages`: массив входных сообщений. Наши модели обучены работать с чередующимися разговорными оборотами «user» и «assistant». При создании нового `Message` вы указываете предыдущие диалоговые ходы с помощью параметра messages, а затем модель генерирует следующий `Message` в диалоге.
  - Каждое входное сообщение должно быть объектом с `role` и `content`. Вы можете указать одно сообщение с ролью `user` или включить несколько сообщений `user` и `assistant` (в этом случае они должны чередоваться). В первом сообщении всегда должен использоваться пользователь «role».

Есть также необязательные параметры, такие как:
- `system`: системное приглашение — подробнее об этом ниже.
  
- `temperature`: степень изменчивости ответа Клода. Для этих уроков и упражнений мы установили для temperature значение 0.

Полный список всех параметров API можно найти в нашей [документации по API](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-claude.html).

### Примеры

Давайте посмотрим, как Клод отвечает на некоторые правильно отформатированные запросы. Для каждой из следующих ячеек запустите ячейку (`shift+enter`), и ответ Клода появится под блоком.

In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

Теперь давайте посмотрим на некоторые подсказки, которые не содержат правильного форматирования API сообщений. Для этих неверных запросов API сообщений возвращает ошибку.

Во-первых, у нас есть пример вызова API сообщений, в котором отсутствуют поля role и content в массиве messages.

> ⚠️ **Внимание:** из-за неправильного форматирования параметра messages в приглашении следующая ячейка вернет ошибку. Это ожидаемое поведение.

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

Вот подсказка, в которой не удается чередовать роли user и assistant.

> ⚠️ **Внимание:** Из-за отсутствия чередования ролей `user` и `assistant`, Клод вернет сообщение об ошибке. Это ожидаемое поведение.

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

Сообщения `user` и `assistant` **ДОЛЖНЫ чередоваться**, а сообщения **ДОЛЖНЫ начинаться с поворота `user`**. В приглашении вы можете использовать несколько пар user и assistant (как будто имитируя многоходовой разговор). Вы также можете вставить слова в сообщение терминала `assistant`, чтобы Клод мог продолжить с того места, где вы остановились (подробнее об этом в последующих главах).

#### Системные подсказки

Вы также можете использовать **системные подсказки**. Системная подсказка – это способ **предоставить Клоду контекст, инструкции и указания** перед тем, как задать ему вопрос или задачу в ходе «Пользовательского» хода. 

Структурно системные запросы существуют отдельно от списка сообщений user и assistant и, таким образом, относятся к отдельному параметру system (см. структуру вспомогательной функции get_completion в разделе [Setup](#setup) записной книжки). 

В этом руководстве, где бы мы ни использовали системную подсказку, мы предоставили вам поле «system» в вашей функции завершения. Если вы не хотите использовать системную подсказку, просто установите для переменной SYSTEM_PROMPT пустую строку.

#### Пример системного запроса

In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

Зачем использовать системную подсказку? **Хорошо написанная системная подсказка может улучшить производительность Клода** различными способами, например повысить способность Клода следовать правилам и инструкциям. Для получения дополнительной информации посетите нашу документацию по [как использовать системные подсказки](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts) с Клодом.

Теперь мы углубимся в некоторые упражнения. Если вы хотите поэкспериментировать с подсказками к уроку, не изменяя какое-либо содержание выше, прокрутите блокнот урока до самого конца, чтобы перейти к [**Пример игровой площадки**](#example-playground).

---

## Упражнения
- [Упражнение 1.1 – Счет до трёх](#exercise-11---counting-to-three)
- [Упражнение 1.2. Системная подсказка](#exercise-12---system-prompt)

### Упражнение 1.1 — Счет до трех
Используя правильное форматирование `user` / `assistant`, отредактируйте `PROMPT` ниже, чтобы Клод считал **досчёт до трёх.** В выводе также будет указано, правильно ли ваше решение.

In [ ]:
# Prompt - this is the only field you should change
PROMPT = "[Replace this text]"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓Если хотите подсказку, пробегите ячейку ниже!

In [ ]:
print(hints.exercise_1_1_hint)

### Упражнение 1.2. Системная подсказка

Измените `SYSTEM_PROMPT`, чтобы Клод реагировал так, будто это трехлетний ребенок.

In [ ]:
# System prompt - this is the only field you should change
SYSTEM_PROMPT = "[Replace this text]"

# Prompt
PROMPT = "How big is the sky?"

# Get Claude's response
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓Если хотите подсказку, пробегите ячейку ниже!

In [ ]:
print(hints.exercise_1_2_hint)

### Поздравляю!

Если вы выполнили все упражнения до этого момента, вы готовы перейти к следующей главе. Приятного подсказки!

---

## Пример игровой площадки

Это область, где вы можете свободно экспериментировать с примерами подсказок, представленными в этом уроке, и настраивать подсказки, чтобы увидеть, как они могут повлиять на ответы Клода.

In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))